# Demo — Google Cloud Knowledge Catalog

Notebook de demonstração do **Google Cloud Knowledge Catalog** (evolução do Dataplex Universal Catalog) para rodar no **Colab Enterprise**.

> ⚠️ **Todos os dados são sintéticos/fake.** Nenhuma informação real de cliente é usada.

📖 [Documentação oficial](https://docs.cloud.google.com/dataplex/docs/introduction) · 📰 [Blog post de lançamento](https://cloud.google.com/blog/products/data-analytics/introducing-the-google-cloud-knowledge-catalog)

## O que esta demo cobre

| # | Etapa | Funcionalidade |
|---|-------|----------------|
| 1 | Tabelas fake | Criação de dados sintéticos no BigQuery ⭐ |
| 2 | Descoberta & busca | Ingestão automática de metadados + busca semântica |
| 3 | Documentação com Gemini | Geração de descrições para o catálogo ⭐ |
| 4 | Aspects | Metadados customizados (aspect types) |
| 5 | Glossário de negócio | Glossário, categorias, termos e associação a colunas ⭐ |
| 6 | Relacionamentos | Data relationships entre tabelas ⭐ |
| 7 | Data profiling | Perfil estatístico automatizado |
| 8 | Auto data quality | Regras de qualidade e scans |
| 9 | Data lineage | Linhagem automática de dados |
| 10 | AI grounding | Recuperação de contexto para agentes (MCP / APIs) |
| 11 | Demais recursos | Data products, insights, import/export, change feeds |

⭐ = etapas destacadas da demo.


## Setup

Instala dependências, define parâmetros e habilita as APIs necessárias.

**Ajuste `PROJECT_ID` se necessário** — por padrão usa o projeto ativo do ambiente.

In [ ]:
%pip install -q --upgrade google-cloud-dataplex google-cloud-bigquery \
    google-cloud-datacatalog-lineage google-genai pandas pyarrow db-dtypes

In [ ]:
import google.auth

credentials, detected_project = google.auth.default()

PROJECT_ID = detected_project          # ou defina manualmente: "meu-projeto"
LOCATION = "us-central1"               # região para data scans e Gemini
BQ_LOCATION = "US"                     # multi-região do dataset BigQuery
CATALOG_LOCATION = "us"                # entradas de tabelas BQ ficam na multi-região "us"
DATASET_ID = "kc_demo"
GLOSSARY_ID = "kc-demo-glossary"

print(f"Projeto: {PROJECT_ID}")

In [ ]:
# Habilita as APIs necessárias (idempotente)
!gcloud services enable dataplex.googleapis.com bigquery.googleapis.com \
    datacatalog.googleapis.com datalineage.googleapis.com \
    aiplatform.googleapis.com --project {PROJECT_ID}

In [ ]:
# Helper para chamar a API REST do Dataplex/Knowledge Catalog
# (algumas funcionalidades em Preview ainda não estão nas client libraries)
import json
import time
from google.auth.transport.requests import AuthorizedSession

session = AuthorizedSession(credentials)
DATAPLEX = "https://dataplex.googleapis.com/v1"


def dataplex_api(method, path, body=None, params=None):
    resp = session.request(method, f"{DATAPLEX}/{path}", json=body, params=params)
    if resp.status_code >= 300:
        raise RuntimeError(f"HTTP {resp.status_code}: {resp.text}")
    return resp.json() if resp.text else {}


def wait_operation(op, timeout=300):
    """Aguarda uma long-running operation da API do Dataplex."""
    name = op.get("name")
    if not name or op.get("done"):
        return op
    deadline = time.time() + timeout
    while time.time() < deadline:
        op = dataplex_api("GET", name)
        if op.get("done"):
            if "error" in op:
                raise RuntimeError(op["error"])
            return op.get("response", op)
        time.sleep(5)
    raise TimeoutError(name)

## Etapa 1 ⭐ — Criação das tabelas fake no BigQuery

Cria o dataset `kc_demo` com 5 tabelas de um e-commerce fictício:

- `customers` — clientes sintéticos
- `products` — catálogo de produtos
- `orders` — pedidos
- `order_items` — itens dos pedidos
- `support_tickets` — tickets de suporte

As chaves entre tabelas (`customer_id`, `order_id`, `product_id`) são o que permitirá ao
Knowledge Catalog descobrir **relacionamentos** mais adiante.

In [ ]:
import random
import datetime
import pandas as pd
from google.cloud import bigquery

random.seed(42)
bq = bigquery.Client(project=PROJECT_ID, location=BQ_LOCATION)

bq.create_dataset(bigquery.Dataset(f"{PROJECT_ID}.{DATASET_ID}"), exists_ok=True)

NOMES = ["Ana", "Bruno", "Carla", "Diego", "Elisa", "Fabio", "Gabriela", "Henrique",
         "Isabela", "Joao", "Karina", "Lucas", "Mariana", "Nicolas", "Olivia", "Pedro"]
SOBRENOMES = ["Silva", "Souza", "Oliveira", "Santos", "Pereira", "Costa", "Almeida",
              "Ferreira", "Rodrigues", "Lima", "Gomes", "Martins"]
CIDADES = [("Sao Paulo", "SP"), ("Rio de Janeiro", "RJ"), ("Belo Horizonte", "MG"),
           ("Curitiba", "PR"), ("Porto Alegre", "RS"), ("Salvador", "BA"),
           ("Recife", "PE"), ("Fortaleza", "CE"), ("Brasilia", "DF")]
SEGMENTOS = ["VAREJO", "ATACADO", "CORPORATIVO"]
CATEGORIAS = ["Eletronicos", "Casa e Cozinha", "Esportes", "Livros", "Moda", "Brinquedos"]
STATUS_PEDIDO = ["PENDENTE", "PAGO", "ENVIADO", "ENTREGUE", "CANCELADO"]

hoje = datetime.date(2026, 7, 1)

def cidade_aleatoria():
    return random.choice(CIDADES)

customers = pd.DataFrame([{
    "customer_id": i,
    "nome": f"{random.choice(NOMES)} {random.choice(SOBRENOMES)}",
    "email": f"cliente{i}@example.com",
    **dict(zip(["cidade", "estado"], cidade_aleatoria())),
    "segmento": random.choice(SEGMENTOS),
    "data_cadastro": hoje - datetime.timedelta(days=random.randint(30, 1500)),
} for i in range(1, 501)])

products = pd.DataFrame([{
    "product_id": i,
    "nome_produto": f"Produto Demo {i:03d}",
    "categoria": random.choice(CATEGORIAS),
    "preco": round(random.uniform(9.9, 2999.9), 2),
} for i in range(1, 51)])

orders = pd.DataFrame([{
    "order_id": i,
    "customer_id": random.randint(1, 500),
    "data_pedido": hoje - datetime.timedelta(days=random.randint(0, 365)),
    "status": random.choice(STATUS_PEDIDO),
} for i in range(1, 2001)])

order_items = pd.DataFrame([{
    "item_id": i,
    "order_id": random.randint(1, 2000),
    "product_id": random.randint(1, 50),
    "quantidade": random.randint(1, 5),
} for i in range(1, 5001)])
order_items["preco_unitario"] = order_items["product_id"].map(
    products.set_index("product_id")["preco"])
totais = (order_items.assign(total=lambda d: d.quantidade * d.preco_unitario)
          .groupby("order_id")["total"].sum())
orders["valor_total"] = orders["order_id"].map(totais).fillna(0).round(2)

support_tickets = pd.DataFrame([{
    "ticket_id": i,
    "customer_id": random.randint(1, 500),
    "order_id": random.randint(1, 2000),
    "assunto": random.choice(["Atraso na entrega", "Produto com defeito", "Troca",
                              "Duvida de cobranca", "Cancelamento"]),
    "prioridade": random.choice(["BAIXA", "MEDIA", "ALTA"]),
    "status": random.choice(["ABERTO", "EM_ANDAMENTO", "RESOLVIDO"]),
    "data_abertura": hoje - datetime.timedelta(days=random.randint(0, 180)),
} for i in range(1, 301)])

tabelas = {"customers": customers, "products": products, "orders": orders,
           "order_items": order_items, "support_tickets": support_tickets}
job_config = bigquery.LoadJobConfig(write_disposition="WRITE_TRUNCATE")
for nome, df in tabelas.items():
    bq.load_table_from_dataframe(df, f"{PROJECT_ID}.{DATASET_ID}.{nome}",
                                 job_config=job_config).result()
    print(f"✓ {DATASET_ID}.{nome}: {len(df)} linhas")

### Consultas com JOIN

Executa consultas com `JOIN` entre as tabelas. Isso alimenta duas funcionalidades:

1. **Data relationships** — o Knowledge Catalog analisa os logs de consulta para descobrir padrões de `JOIN` (pode levar **até 48h**);
2. **Data lineage** — as tabelas criadas via `CREATE TABLE AS SELECT` geram linhagem automática.

In [ ]:
consultas = [
    # CTAS -> gera lineage
    f"""CREATE OR REPLACE TABLE `{PROJECT_ID}.{DATASET_ID}.vendas_por_cliente` AS
        SELECT c.customer_id, c.nome, c.segmento, c.estado,
               COUNT(o.order_id) AS qtd_pedidos, SUM(o.valor_total) AS receita_total
        FROM `{PROJECT_ID}.{DATASET_ID}.customers` c
        JOIN `{PROJECT_ID}.{DATASET_ID}.orders` o ON o.customer_id = c.customer_id
        GROUP BY 1, 2, 3, 4""",
    f"""CREATE OR REPLACE TABLE `{PROJECT_ID}.{DATASET_ID}.receita_por_categoria` AS
        SELECT p.categoria, SUM(i.quantidade * i.preco_unitario) AS receita
        FROM `{PROJECT_ID}.{DATASET_ID}.order_items` i
        JOIN `{PROJECT_ID}.{DATASET_ID}.products` p ON p.product_id = i.product_id
        GROUP BY 1""",
    # SELECTs com JOIN -> sinal para descoberta de relacionamentos
    f"""SELECT o.order_id, c.nome, o.valor_total
        FROM `{PROJECT_ID}.{DATASET_ID}.orders` o
        JOIN `{PROJECT_ID}.{DATASET_ID}.customers` c ON c.customer_id = o.customer_id
        LIMIT 10""",
    f"""SELECT i.order_id, p.nome_produto, i.quantidade
        FROM `{PROJECT_ID}.{DATASET_ID}.order_items` i
        JOIN `{PROJECT_ID}.{DATASET_ID}.products` p ON p.product_id = i.product_id
        LIMIT 10""",
    f"""SELECT t.ticket_id, c.nome, o.status AS status_pedido
        FROM `{PROJECT_ID}.{DATASET_ID}.support_tickets` t
        JOIN `{PROJECT_ID}.{DATASET_ID}.customers` c ON c.customer_id = t.customer_id
        JOIN `{PROJECT_ID}.{DATASET_ID}.orders` o ON o.order_id = t.order_id
        LIMIT 10""",
]

for rodada in range(3):  # repete para reforçar o sinal nos logs de consulta
    for sql in consultas:
        bq.query(sql).result()
    print(f"✓ Rodada {rodada + 1} de consultas executada")

## Etapa 2 — Descoberta automática e busca semântica

As tabelas do BigQuery são **catalogadas automaticamente** — sem nenhuma configuração.
A busca do Knowledge Catalog suporta busca por palavra-chave, filtros facetados e
**busca semântica em linguagem natural** (GA, baseada em tecnologia do Google Search).

[Docs de busca](https://docs.cloud.google.com/dataplex/docs/search-assets)

In [ ]:
# Busca por palavra-chave
res = dataplex_api("POST", f"projects/{PROJECT_ID}/locations/global:searchEntries",
                   body={"query": f"parent:{DATASET_ID}", "pageSize": 20})
print(f"Entradas encontradas: {res.get('totalSize', 0)}")
for r in res.get("results", []):
    e = r.get("dataplexEntry", {})
    print(" -", e.get("entrySource", {}).get("displayName") or e.get("name"))

In [ ]:
# Busca semântica em linguagem natural
res = dataplex_api("POST", f"projects/{PROJECT_ID}/locations/global:searchEntries",
                   body={"query": "tabelas de pedidos e vendas de clientes",
                         "semanticSearch": True, "pageSize": 5})
for r in res.get("results", []):
    e = r.get("dataplexEntry", {})
    print(" -", e.get("entrySource", {}).get("displayName") or e.get("name"))

In [ ]:
# Inspeciona a entrada da tabela `orders` no catálogo (com aspects de schema)
def entry_name(table):
    return (f"projects/{PROJECT_ID}/locations/{CATALOG_LOCATION}/entryGroups/@bigquery/"
            f"entries/bigquery.googleapis.com/projects/{PROJECT_ID}/datasets/"
            f"{DATASET_ID}/tables/{table}")

entry = dataplex_api("GET", f"projects/{PROJECT_ID}/locations/{CATALOG_LOCATION}:lookupEntry",
                     params={"entry": entry_name("orders"), "view": "ALL"})
print("Entry:", entry["name"])
print("Tipo:", entry.get("entryType"))
print("Aspects:", list(entry.get("aspects", {}).keys()))

## Etapa 3 ⭐ — Documentação do catálogo gerada com Gemini

Gera descrições de tabelas e colunas com **Gemini** e grava no BigQuery.
As descrições sincronizam automaticamente com o Knowledge Catalog.

> No console, o recurso equivalente é a *curadoria automática de contexto* (Preview),
> que gera descrições, glossários e relacionamentos automaticamente.

In [ ]:
from google import genai

gemini = genai.Client(vertexai=True, project=PROJECT_ID, location="global")
MODEL = "gemini-2.5-flash"


def gerar_documentacao(table_ref):
    table = bq.get_table(table_ref)
    schema_txt = "\n".join(f"- {f.name} ({f.field_type})" for f in table.schema)
    prompt = f"""Você é um data steward. Gere documentação em português para a tabela
BigQuery `{table.table_id}` de um e-commerce (dados sintéticos de demonstração).

Schema:
{schema_txt}

Responda APENAS com JSON válido no formato:
{{"table_description": "...", "columns": {{"nome_coluna": "descrição", ...}}}}
Descrições concisas (1 frase), tom profissional."""
    resp = gemini.models.generate_content(model=MODEL, contents=prompt)
    texto = resp.text.strip().removeprefix("```json").removeprefix("```").removesuffix("```")
    return json.loads(texto)


for nome in ["customers", "products", "orders", "order_items", "support_tickets"]:
    ref = f"{PROJECT_ID}.{DATASET_ID}.{nome}"
    docs = gerar_documentacao(ref)
    table = bq.get_table(ref)
    table.description = docs["table_description"]
    novo_schema = [
        bigquery.SchemaField(f.name, f.field_type, mode=f.mode,
                             description=docs["columns"].get(f.name, f.description))
        for f in table.schema
    ]
    table.schema = novo_schema
    bq.update_table(table, ["description", "schema"])
    print(f"✓ {nome}: {docs['table_description'][:90]}...")

In [ ]:
# Adiciona também um "overview" rico (aspect padrão do catálogo) na tabela orders
overview_html = (
    "<p><b>Tabela de pedidos do e-commerce demo (dados sintéticos).</b></p>"
    "<p>Fatos de pedidos com valor total, status e data. "
    "Relaciona-se com <i>customers</i> (customer_id), <i>order_items</i> (order_id) "
    "e <i>support_tickets</i> (order_id).</p>"
    "<p>Documentação gerada automaticamente na demo do Knowledge Catalog.</p>"
)
try:
    dataplex_api("PATCH", entry_name("orders"),
                 params={"updateMask": "aspects"},
                 body={"aspects": {
                     "dataplex-types.global.overview": {"data": {"content": overview_html}}
                 }})
    print("✓ Overview adicionado à entrada da tabela orders")
except Exception as e:
    print("Overview via API falhou (adicione pelo console se necessário):", e)

## Etapa 4 — Aspects: metadados customizados

**Aspect types** definem templates de metadados estruturados que podem ser anexados a
qualquer entrada do catálogo — ex.: dono do dado, domínio, sensibilidade.

[Docs de aspects](https://docs.cloud.google.com/dataplex/docs/enrich-entries-metadata)

In [ ]:
ASPECT_TYPE_ID = "governanca-demo"
aspect_type_name = f"projects/{PROJECT_ID}/locations/global/aspectTypes/{ASPECT_TYPE_ID}"

try:
    op = dataplex_api("POST", f"projects/{PROJECT_ID}/locations/global/aspectTypes",
                      params={"aspectTypeId": ASPECT_TYPE_ID},
                      body={
                          "displayName": "Governança (demo)",
                          "description": "Metadados de governança da demo",
                          "metadataTemplate": {
                              "name": "governanca-demo",
                              "type": "record",
                              "recordFields": [
                                  {"name": "owner_email", "type": "string", "index": 1,
                                   "annotations": {"displayName": "E-mail do dono"}},
                                  {"name": "dominio", "type": "string", "index": 2,
                                   "annotations": {"displayName": "Domínio de dados"}},
                                  {"name": "sensibilidade", "type": "enum", "index": 3,
                                   "annotations": {"displayName": "Sensibilidade"},
                                   "enumValues": [{"name": "PUBLICO", "index": 1},
                                                  {"name": "INTERNO", "index": 2},
                                                  {"name": "CONFIDENCIAL", "index": 3}]},
                              ],
                          },
                      })
    wait_operation(op)
    print("✓ Aspect type criado")
except RuntimeError as e:
    print("Aspect type já existe" if "409" in str(e) else f"Erro: {e}")

In [ ]:
# Anexa o aspect de governança às tabelas
aspect_key = f"{PROJECT_ID}.global.{ASPECT_TYPE_ID}"
for nome in ["customers", "orders", "products"]:
    dataplex_api("PATCH", entry_name(nome),
                 params={"updateMask": "aspects"},
                 body={"aspects": {aspect_key: {"data": {
                     "owner_email": "data-team@example.com",
                     "dominio": "vendas",
                     "sensibilidade": "INTERNO",
                 }}}})
    print(f"✓ Aspect anexado a {nome}")

## Etapa 5 ⭐ — Glossário de negócio

Cria um glossário com categorias e termos de negócio, e **associa termos a colunas**
das tabelas via entry links (tipo `definition`).

[Docs de glossário](https://docs.cloud.google.com/dataplex/docs/manage-glossaries)

In [ ]:
glossary_name = f"projects/{PROJECT_ID}/locations/global/glossaries/{GLOSSARY_ID}"

try:
    op = dataplex_api("POST", f"projects/{PROJECT_ID}/locations/global/glossaries",
                      params={"glossaryId": GLOSSARY_ID},
                      body={"displayName": "Glossário E-commerce (demo)",
                            "description": "Terminologia de negócio da demo do Knowledge Catalog"})
    wait_operation(op)
    print("✓ Glossário criado")
except RuntimeError as e:
    print("Glossário já existe" if "409" in str(e) else f"Erro: {e}")

# Categorias
categorias = {
    "vendas": "Termos relacionados ao processo de vendas",
    "atendimento": "Termos relacionados ao atendimento ao cliente",
}
for cat_id, desc in categorias.items():
    try:
        dataplex_api("POST", f"{glossary_name}/categories",
                     params={"categoryId": cat_id},
                     body={"displayName": cat_id.capitalize(), "description": desc,
                           "parent": glossary_name})
        print(f"✓ Categoria: {cat_id}")
    except RuntimeError as e:
        print(f"Categoria {cat_id} já existe" if "409" in str(e) else f"Erro: {e}")

In [ ]:
termos = {
    "cliente": ("Cliente", "Pessoa física ou jurídica que realiza compras na plataforma.", "vendas"),
    "pedido": ("Pedido", "Solicitação de compra feita por um cliente, composta por um ou mais itens.", "vendas"),
    "receita-total": ("Receita Total", "Soma do valor de todos os itens de um pedido, em BRL.", "vendas"),
    "segmento": ("Segmento", "Classificação comercial do cliente: VAREJO, ATACADO ou CORPORATIVO.", "vendas"),
    "ticket-suporte": ("Ticket de Suporte", "Registro de atendimento aberto por um cliente.", "atendimento"),
}
for term_id, (nome, desc, cat) in termos.items():
    try:
        dataplex_api("POST", f"{glossary_name}/terms",
                     params={"termId": term_id},
                     body={"displayName": nome, "description": desc,
                           "parent": f"{glossary_name}/categories/{cat}"})
        print(f"✓ Termo: {nome}")
    except RuntimeError as e:
        print(f"Termo {nome} já existe" if "409" in str(e) else f"Erro: {e}")

In [ ]:
# Associa termos do glossário a colunas das tabelas (entry link tipo "definition")
def term_entry(term_id):
    return (f"projects/{PROJECT_ID}/locations/global/entryGroups/@dataplex/entries/"
            f"projects/{PROJECT_ID}/locations/global/glossaries/{GLOSSARY_ID}/terms/{term_id}")

associacoes = [
    # (termo, tabela, coluna) — coluna None associa à tabela inteira
    ("cliente", "customers", None),
    ("pedido", "orders", None),
    ("receita-total", "orders", "valor_total"),
    ("segmento", "customers", "segmento"),
    ("ticket-suporte", "support_tickets", None),
]

for i, (term_id, tabela, coluna) in enumerate(associacoes):
    source = {"name": entry_name(tabela), "type": "SOURCE"}
    if coluna:
        source["path"] = coluna
    try:
        dataplex_api(
            "POST",
            f"projects/{PROJECT_ID}/locations/{CATALOG_LOCATION}/entryGroups/@bigquery/entryLinks",
            params={"entryLinkId": f"kc-demo-def-{i}"},
            body={
                "entryLinkType": "projects/dataplex-types/locations/global/entryLinkTypes/definition",
                "entryReferences": [source, {"name": term_entry(term_id), "type": "TARGET"}],
            })
        alvo = f"{tabela}.{coluna}" if coluna else tabela
        print(f"✓ {term_id} → {alvo}")
    except RuntimeError as e:
        print(f"Link {term_id} já existe" if "409" in str(e) else
              f"Falha ao vincular {term_id} (associe pelo console): {e}")

> 💡 No console: **Knowledge Catalog → Glossaries** para navegar pelo glossário, e a página
> de cada tabela mostra os termos associados. O glossário também suporta
> **import/export via JSON e Google Sheets**.

## Etapa 6 ⭐ — Relacionamentos entre tabelas (data relationships)

O Knowledge Catalog descobre relacionamentos entre tabelas do BigQuery de duas formas
([docs](https://docs.cloud.google.com/dataplex/docs/data-relationships), Preview):

1. **Análise de logs de consulta** — padrões de `JOIN` detectados continuamente
   (⏱️ pode levar **até 48h** após as consultas da Etapa 1; apenas uma amostra dos logs é analisada);
2. **Data insights** — sugestões sob demanda via LLM, analisando schema e sobreposição de dados
   (no console, na página da tabela).

Os relacionamentos ficam armazenados como **entry links** entre entradas do catálogo e
alimentam agentes de IA com contexto estrutural (JOINs corretos, sem "adivinhação").

In [ ]:
# Consulta os relacionamentos descobertos para a tabela orders
# (se rodar logo após criar as tabelas, a lista estará vazia — aguarde a descoberta, até 48h)
try:
    links = dataplex_api(
        "GET",
        f"projects/{PROJECT_ID}/locations/{CATALOG_LOCATION}:lookupEntryLinks",
        params={"entry": entry_name("orders")})
    itens = links.get("entryLinks", [])
    if not itens:
        print("Nenhum relacionamento descoberto ainda — a análise de logs leva até 48h.")
    for link in itens:
        refs = link.get("entryReferences", [])
        tipo = link.get("entryLinkType", "").split("/")[-1]
        print(f"[{tipo}] " + " ↔ ".join(r.get("name", "").split("/")[-1] for r in refs))
except Exception as e:
    print("API de lookup indisponível/Preview — consulte pelo console:", e)

print("""
📍 No console: Knowledge Catalog → Search → tabela `orders` → aba **Relationships**
   (mostra Target, colunas do relacionamento, Type=JOIN, Origin e uma query de exemplo).
📍 Sugestões sob demanda: na página da tabela, use **Data insights** para o LLM sugerir
   relacionamentos com base no schema.""")

## Etapa 7 — Data profiling

Cria e executa um **profile scan** na tabela `orders`: estatísticas por coluna
(nulos, distintos, min/max/média, top values).

[Docs de profiling](https://docs.cloud.google.com/dataplex/docs/data-profiling-overview)

In [ ]:
from google.cloud import dataplex_v1

scan_client = dataplex_v1.DataScanServiceClient()
scan_parent = f"projects/{PROJECT_ID}/locations/{LOCATION}"


def criar_e_rodar_scan(scan_id, scan):
    try:
        scan_client.create_data_scan(parent=scan_parent, data_scan=scan,
                                     data_scan_id=scan_id).result()
        print(f"✓ Scan {scan_id} criado")
    except Exception:
        print(f"Scan {scan_id} já existe — reutilizando")
    run = scan_client.run_data_scan(name=f"{scan_parent}/dataScans/{scan_id}")
    job_name = run.job.name
    while True:
        job = scan_client.get_data_scan_job(
            request={"name": job_name,
                     "view": dataplex_v1.GetDataScanJobRequest.DataScanJobView.FULL})
        if job.state in (dataplex_v1.DataScanJob.State.SUCCEEDED,
                         dataplex_v1.DataScanJob.State.FAILED,
                         dataplex_v1.DataScanJob.State.CANCELLED):
            return job
        time.sleep(15)


profile_scan = dataplex_v1.DataScan()
profile_scan.data.resource = (f"//bigquery.googleapis.com/projects/{PROJECT_ID}"
                              f"/datasets/{DATASET_ID}/tables/orders")
profile_scan.data_profile_spec = dataplex_v1.DataProfileSpec()
profile_scan.execution_spec.trigger.on_demand = {}

job = criar_e_rodar_scan("kc-demo-profile-orders", profile_scan)
print(f"Estado: {job.state.name}\n")
for campo in job.data_profile_result.profile.fields:
    p = campo.profile
    print(f"{campo.name:15s} nulos={p.null_ratio:.1%}  distintos={p.distinct_ratio:.1%}")

## Etapa 8 — Auto data quality

Cria um **data quality scan** com regras declarativas na tabela `orders`:
completude, unicidade, valores válidos e faixa numérica. Os resultados aparecem
também na página da tabela no console.

[Docs de auto DQ](https://docs.cloud.google.com/dataplex/docs/auto-data-quality-overview)

In [ ]:
Rule = dataplex_v1.DataQualityRule

regras = [
    Rule(column="order_id", dimension="COMPLETENESS", threshold=1.0,
         non_null_expectation=Rule.NonNullExpectation()),
    Rule(column="order_id", dimension="UNIQUENESS",
         uniqueness_expectation=Rule.UniquenessExpectation()),
    Rule(column="status", dimension="VALIDITY", threshold=1.0,
         set_expectation=Rule.SetExpectation(
             values=["PENDENTE", "PAGO", "ENVIADO", "ENTREGUE", "CANCELADO"])),
    Rule(column="valor_total", dimension="VALIDITY", threshold=0.95,
         range_expectation=Rule.RangeExpectation(min_value="0")),
]

dq_scan = dataplex_v1.DataScan()
dq_scan.data.resource = (f"//bigquery.googleapis.com/projects/{PROJECT_ID}"
                         f"/datasets/{DATASET_ID}/tables/orders")
dq_scan.data_quality_spec = dataplex_v1.DataQualitySpec(rules=regras)
dq_scan.execution_spec.trigger.on_demand = {}

job = criar_e_rodar_scan("kc-demo-dq-orders", dq_scan)
r = job.data_quality_result
print(f"Resultado geral: {'✅ PASSOU' if r.passed else '❌ FALHOU'}  score={r.score:.1f}\n")
for regra in r.rules:
    status = "✅" if regra.passed else "❌"
    print(f"{status} {regra.rule.column:12s} {regra.rule.dimension:13s} "
          f"aprovação={regra.pass_ratio:.1%}")

> 💡 Regras também podem ser geridas como código (**Terraform / Cloud Build / GitHub**)
> e agendadas com triggers recorrentes.

## Etapa 9 — Data lineage

As tabelas `vendas_por_cliente` e `receita_por_categoria` foram criadas via
`CREATE TABLE AS SELECT` — o BigQuery registra a **linhagem automaticamente**.

[Docs de lineage](https://docs.cloud.google.com/dataplex/docs/about-data-lineage)
· Suporta também OpenLineage e sistemas externos.

In [ ]:
from google.cloud import datacatalog_lineage_v1

lineage = datacatalog_lineage_v1.LineageClient()
alvo = datacatalog_lineage_v1.EntityReference(
    fully_qualified_name=f"bigquery:{PROJECT_ID}.{DATASET_ID}.vendas_por_cliente")

req = datacatalog_lineage_v1.SearchLinksRequest(
    parent=f"projects/{PROJECT_ID}/locations/us", target=alvo)
encontrou = False
try:
    for link in lineage.search_links(request=req):
        encontrou = True
        print(f"{link.source.fully_qualified_name}  →  {link.target.fully_qualified_name}")
except Exception as e:
    print("Erro na consulta de lineage:", e)
if not encontrou:
    print("Lineage ainda não disponível — aguarde alguns minutos e rode novamente.")

print("""
📍 No console: página da tabela `vendas_por_cliente` → aba **Lineage** (grafo visual).
Casos de uso: análise de impacto, rastreamento de PII, otimização de custos.""")

## Etapa 10 — AI grounding: contexto para agentes de IA

O Knowledge Catalog serve contexto empresarial para agentes de IA — o coração do
posicionamento de *universal context engine*:

- **Servidores MCP** (local e remoto) — agentes acessam o catálogo via Model Context Protocol
  ([docs](https://docs.cloud.google.com/dataplex/docs/mcp-overview));
- **APIs de recuperação de contexto** — ex.: `LookupContext` retorna um pacote de metadados
  pré-formatado (schema, descrições, termos de glossário, relacionamentos) para grounding
  ([docs](https://docs.cloud.google.com/dataplex/docs/retrieve-data-context));
- **Queries verificadas e guardrails semânticos** (Preview) — padrões SQL verificados evitam
  JOINs "alucinados";
- **Context Graph** — mapa dinâmico conectando schemas, entidades de negócio e conhecimento
  não estruturado.

Tudo o que fizemos nas etapas anteriores (documentação, glossário, relacionamentos,
qualidade) **é exatamente o contexto** que essas APIs entregam aos agentes.

In [ ]:
# Recupera o pacote de contexto da tabela orders (API em Preview — pode variar)
try:
    ctx = dataplex_api(
        "GET",
        f"projects/{PROJECT_ID}/locations/{CATALOG_LOCATION}:lookupContext",
        params={"entry": entry_name("orders")})
    print(json.dumps(ctx, indent=2, ensure_ascii=False)[:3000])
except Exception as e:
    print("LookupContext em Preview — se indisponível no seu projeto, use lookupEntry:\n")
    entry = dataplex_api("GET",
                         f"projects/{PROJECT_ID}/locations/{CATALOG_LOCATION}:lookupEntry",
                         params={"entry": entry_name("orders"), "view": "ALL"})
    src = entry.get("entrySource", {})
    print(f"Tabela: {src.get('displayName')}")
    print(f"Descrição: {src.get('description', '')[:200]}")
    print(f"Aspects disponíveis: {list(entry.get('aspects', {}).keys())}")

## Etapa 11 — Demais funcionalidades (console / Preview)

Recursos que completam o Knowledge Catalog, majoritariamente via console ou em Preview:

- **Data products (GA)** — empacote as tabelas da demo em um produto de dados com SLA e
  restrições de governança: console → *Knowledge Catalog → Data products*
  ([docs](https://docs.cloud.google.com/dataplex/docs/data-products-overview));
- **Data insights** — queries geradas por Gemini na aba *Insights* de cada tabela;
- **Insights de dados não estruturados (Preview)** — extração multimodal de metadados de
  PDFs/arquivos no Cloud Storage com Gemini
  ([docs](https://docs.cloud.google.com/dataplex/docs/data-insights-unstructured-data));
- **Smart Storage / Object Context API (Preview)** — enriquecimento automático de objetos
  no Cloud Storage no momento do upload;
- **Import/export de metadados** — pipelines de conectividade gerenciada para Oracle, MySQL,
  PostgreSQL, SQL Server e catálogos parceiros (Atlan, Collibra, Datahub…)
  ([docs](https://docs.cloud.google.com/dataplex/docs/managed-connectivity-overview));
- **Metadata change feeds** — notificações de mudança de metadados via Pub/Sub
  ([docs](https://docs.cloud.google.com/dataplex/docs/metadata-change-feeds-overview));
- **BigQuery measures + LookML (Preview)** — semântica de negócio governada no engine SQL;
- **Segurança** — IAM, VPC-SC, CMEK; a busca respeita as permissões dos sistemas de origem.

## Limpeza (opcional)

Descomente e execute para remover todos os recursos criados pela demo.

In [ ]:
# --- descomente para limpar ---
# bq.delete_dataset(f"{PROJECT_ID}.{DATASET_ID}", delete_contents=True, not_found_ok=True)
# for scan_id in ["kc-demo-profile-orders", "kc-demo-dq-orders"]:
#     try:
#         scan_client.delete_data_scan(name=f"{scan_parent}/dataScans/{scan_id}")
#     except Exception as e:
#         print(e)
# try:
#     # remova primeiro os entry links, depois termos/categorias/glossário e o aspect type
#     for i in range(5):
#         dataplex_api("DELETE", f"projects/{PROJECT_ID}/locations/{CATALOG_LOCATION}"
#                                f"/entryGroups/@bigquery/entryLinks/kc-demo-def-{i}")
#     for t in ["cliente", "pedido", "receita-total", "segmento", "ticket-suporte"]:
#         dataplex_api("DELETE", f"{glossary_name}/terms/{t}")
#     for c in ["vendas", "atendimento"]:
#         dataplex_api("DELETE", f"{glossary_name}/categories/{c}")
#     wait_operation(dataplex_api("DELETE", glossary_name))
#     wait_operation(dataplex_api("DELETE", aspect_type_name))
# except Exception as e:
#     print(e)
# print("✓ Limpeza concluída")

## Recapitulando

| Funcionalidade | O que fizemos |
|---|---|
| Tabelas fake ⭐ | 7 tabelas sintéticas de e-commerce no BigQuery |
| Descoberta & busca | Catalogação automática + busca semântica |
| Documentação ⭐ | Descrições geradas com Gemini + overview no catálogo |
| Aspects | Aspect type de governança anexado às tabelas |
| Glossário ⭐ | Glossário, 2 categorias, 5 termos, associação a colunas |
| Relacionamentos ⭐ | JOINs para descoberta via query logs + lookup de entry links |
| Profiling | Profile scan da tabela orders |
| Data quality | 4 regras de DQ com resultados por dimensão |
| Lineage | Linhagem automática das tabelas CTAS |
| AI grounding | Contexto para agentes via MCP / APIs de contexto |

⏱️ **Lembrete**: relacionamentos por query logs levam **até 48h** para aparecer.
Volte à Etapa 6 depois desse período.

---
*Demo com dados 100% sintéticos — sem vínculo oficial com o Google.
Funcionalidades em Preview sujeitas aos
[Pre-GA Offerings Terms](https://cloud.google.com/terms/service-terms).*